Filtrado de calidad y mapeo de mutaciones a genes

Este notebook:
1. Carga los TSV de Nextclade (2020–2024).
2. Filtra secuencias por calidad (qcStatus == "good").
3. Mapea cada mutación de nucleótido a su gen correspondiente usando el GFF3 de referencia.
4. Guarda un TSV consolidado en data/processed/mutaciones_mapeadas.tsv

### Librerías y rutas

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from Bio import SeqIO
from Bio.Seq import Seq


In [4]:
BASE_DIR = Path.cwd().parent
NEXTCLADE_DIR = BASE_DIR / "results" / "nextclade"
GFF_PATH = BASE_DIR / "data" / "sars-cov-2" / "genome_annotation.gff3"
OUT_DIR = BASE_DIR / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
YEARS = [2020, 2021, 2022, 2023, 2024]

## 1. Cargar los TSV de Nextclade

Cada archivo nextclade_YYYY.tsv corresponde a un año. Los concatenamos en un solo DataFrame.

In [5]:
dfs = []
for year in YEARS:
    fpath = NEXTCLADE_DIR / f"nextclade_{year}.tsv"
    if not fpath.exists():
        print(f"[WARN] No existe: {fpath}")
        continue
    df_year = pd.read_csv(fpath, sep="\t")
    df_year["year"] = year
    dfs.append(df_year)
    print(f"Cargado {year}: {len(df_year)} secuencias")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal de secuencias cargadas: {len(df_all)}")
print(f"Columnas disponibles: {list(df_all.columns)}")

Cargado 2020: 81 secuencias
Cargado 2021: 80 secuencias
Cargado 2022: 80 secuencias
Cargado 2023: 80 secuencias
Cargado 2024: 80 secuencias

Total de secuencias cargadas: 401
Columnas disponibles: ['index', 'seqName', 'clade', 'clade_display', 'clade_who', 'clade_nextstrain', 'partiallyAliased', 'Nextclade_pango', 'qc.overallScore', 'qc.overallStatus', 'totalSubstitutions', 'totalDeletions', 'totalInsertions', 'totalFrameShifts', 'totalMissing', 'totalNonACGTNs', 'totalAminoacidSubstitutions', 'totalAminoacidDeletions', 'totalAminoacidInsertions', 'totalUnknownAa', 'alignmentScore', 'alignmentStart', 'alignmentEnd', 'coverage', 'cdsCoverage', 'isReverseComplement', 'substitutions', 'deletions', 'insertions', 'frameShifts', 'aaSubstitutions', 'aaDeletions', 'aaInsertions', 'privateNucMutations.reversionSubstitutions', 'privateNucMutations.labeledSubstitutions', 'privateNucMutations.unlabeledSubstitutions', 'privateNucMutations.totalReversionSubstitutions', 'privateNucMutations.totalLabe

## Filtrar por calidad

Conservamos secuencias con qcStatus == "good". Si el número resultante es muy bajo (<50% del total), relajamos el filtro a totalMissing < 3000.

In [6]:
# Conteo por estado de QC (nombre correcto: qc.overallStatus)
print("Distribución de qc.overallStatus:")
print(df_all["qc.overallStatus"].value_counts(dropna=False))
print()

n_total = len(df_all)
df_good = df_all[df_all["qc.overallStatus"] == "good"].copy()
n_good = len(df_good)

print(f"Secuencias totales: {n_total}")
print(f"Secuencias 'good': {n_good} ({n_good/n_total*100:.1f}%)")

# Si menos del 50% pasa el filtro, relajamos a qc.missingData.totalMissing < 3000
if n_good < 0.5 * n_total:
    print("\n[Aviso] Menos del 50% pasa el filtro 'good'. Relajando a totalMissing < 3000.")
    df_good = df_all[df_all["qc.missingData.totalMissing"] < 3000].copy()
    print(f"Secuencias tras filtro relajado: {len(df_good)}")

print(f"\nSecuencias finales para análisis: {len(df_good)}")
print("\nDistribución por año:")
print(df_good["year"].value_counts().sort_index())

Distribución de qc.overallStatus:
qc.overallStatus
good        383
mediocre     14
bad           4
Name: count, dtype: int64

Secuencias totales: 401
Secuencias 'good': 383 (95.5%)

Secuencias finales para análisis: 383

Distribución por año:
year
2020    80
2021    73
2022    73
2023    77
2024    80
Name: count, dtype: int64


## Parsear el GFF3 para obtener coordenadas de genes

El GFF3 contiene las anotaciones del genoma de referencia (NC_045512.2). Extraemos las features de tipo gene y guardamos sus coordenadas (start, end, strand) para mapear posiciones.

In [9]:
def parse_gff_genes(gff_path):
    """Parsea el GFF3 de Nextclade usando el atributo gene_name=."""
    genes = []
    with open(gff_path, "r") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split("\t")
            if len(parts) < 9:
                continue
            feature_type = parts[2]
            if feature_type != "gene":
                continue

            start = int(parts[3])
            end = int(parts[4])
            strand = parts[6]
            attributes = parts[8]

            # El nombre está en gene_name=
            name_match = re.search(r"gene_name=([^;]+)", attributes)
            gene_name = name_match.group(1).strip() if name_match else "unknown"

            genes.append({
                "gene": gene_name,
                "start": start,
                "end": end,
                "strand": strand,
                "length": end - start + 1,
            })

    return pd.DataFrame(genes)

gff_genes = parse_gff_genes(GFF_PATH)
print(f"Genes encontrados: {len(gff_genes)}")
print(gff_genes.sort_values("start"))

Genes encontrados: 12
     gene  start    end strand  length
3   ORF1a    266  13468      +   13203
4   ORF1b  13468  21555      +    8088
11      S  21563  25384      +    3822
5   ORF3a  25393  26220      +     828
0       E  26245  26472      +     228
1       M  26523  27191      +     669
6    ORF6  27202  27387      +     186
7   ORF7a  27394  27759      +     366
8   ORF7b  27756  27887      +     132
9    ORF8  27894  28259      +     366
2       N  28274  29533      +    1260
10  ORF9b  28284  28577      +     294


## Parsear las mutaciones de nucleótido y mapearlas a genes

La columna substitutions del TSV de Nextclade contiene mutaciones en formato C241T,C3037T,... 
Parseamos cada una, extraemos la posición y la asignamos al gen que la contiene.

In [11]:
def parse_nt_substitutions(subs_str):
    """Convierte 'C241T,A100G' en [('C', 241, 'T'), ('A', 100, 'G')]."""
    if pd.isna(subs_str) or subs_str == "":
        return []
    result = []
    for sub in subs_str.split(","):
        sub = sub.strip()
        m = re.match(r"^([ACGTN])(\d+)([ACGTN])$", sub)
        if m:
            ref, pos, alt = m.group(1), int(m.group(2)), m.group(3)
            result.append((ref, pos, alt))
    return result


def map_position_to_gene(pos, gff_df):
    """Devuelve el nombre del gen que contiene la posición dada."""
    hits = gff_df[(gff_df["start"] <= pos) & (gff_df["end"] >= pos)]
    if len(hits) == 0:
        return "intergenic"
    # Si hay solapamiento (genes anidados), tomamos el más pequeño (más específico)
    return hits.sort_values("length").iloc[0]["gene"]


# Expandir: una fila por mutación
records = []
for _, row in df_good.iterrows():
    muts = parse_nt_substitutions(row["substitutions"])
    for ref, pos, alt in muts:
        records.append({
            "seqName": row["seqName"],
            "year": row["year"],
            "clade": row.get("clade", np.nan),
            "ref_nt": ref,
            "position": pos,
            "alt_nt": alt,
            "mutation": f"{ref}{pos}{alt}",
            "gene": map_position_to_gene(pos, gff_genes),
        })

df_muts = pd.DataFrame(records)
print(f"Total de mutaciones parseadas: {len(df_muts)}")
print(f"Secuencias con al menos una mutación: {df_muts['seqName'].nunique()}")
print("\nDistribución de mutaciones por gen:")
print(df_muts["gene"].value_counts().head(15))

Total de mutaciones parseadas: 23663
Secuencias con al menos una mutación: 378

Distribución de mutaciones por gen:
gene
S             9693
ORF1a         5551
ORF1b         2545
N             1403
M             1054
intergenic     942
ORF3a          676
ORF6           524
ORF9b          398
ORF7b          363
E              238
ORF7a          205
ORF8            71
Name: count, dtype: int64


## Resumen y guardado

Verificamos el resultado y guardamos el TSV consolidado para los siguientes pasos.

In [12]:
# Resumen: mutaciones únicas por gen
summary = (
    df_muts.groupby("gene")["mutation"]
    .nunique()
    .reset_index()
    .rename(columns={"mutation": "n_unique_mutations"})
    .sort_values("n_unique_mutations", ascending=False)
)
print("Mutaciones únicas por gen (top 15):")
print(summary.head(15).to_string(index=False))

# Guardar resultados
out_muts = OUT_DIR / "mutaciones_mapeadas.tsv"
df_muts.to_csv(out_muts, sep="\t", index=False)
print(f"\nGuardado: {out_muts}")

out_good = OUT_DIR / "secuencias_good.tsv"
df_good.to_csv(out_good, sep="\t", index=False)
print(f"Guardado: {out_good}")

Mutaciones únicas por gen (top 15):
      gene  n_unique_mutations
     ORF1a                 732
     ORF1b                 413
         S                 302
         N                 109
     ORF3a                  90
intergenic                  73
         M                  47
     ORF9b                  30
     ORF7a                  29
      ORF8                  29
      ORF6                  16
         E                  14
     ORF7b                   8

Guardado: /export/space3/users/mariosb/LCG/BIOPYTHON/analisis_var_gene/data/processed/mutaciones_mapeadas.tsv
Guardado: /export/space3/users/mariosb/LCG/BIOPYTHON/analisis_var_gene/data/processed/secuencias_good.tsv


In [13]:
print("Primeras 5 filas del GFF parseado:")
print(gff_genes.head())
print("\nRango de posiciones en el GFF:")
print(f"Start mínimo: {gff_genes['start'].min()}, Start máximo: {gff_genes['start'].max()}")
print(f"End mínimo: {gff_genes['end'].min()}, End máximo: {gff_genes['end'].max()}")

Primeras 5 filas del GFF parseado:
    gene  start    end strand  length
0      E  26245  26472      +     228
1      M  26523  27191      +     669
2      N  28274  29533      +    1260
3  ORF1a    266  13468      +   13203
4  ORF1b  13468  21555      +    8088

Rango de posiciones en el GFF:
Start mínimo: 266, Start máximo: 28284
End mínimo: 13468, End máximo: 29533


In [14]:
test_positions = [241, 3037, 14408, 23403, 25563]
for pos in test_positions:
    gene = map_position_to_gene(pos, gff_genes)
    print(f"Posición {pos}: {gene}")

Posición 241: intergenic
Posición 3037: ORF1a
Posición 14408: ORF1b
Posición 23403: S
Posición 25563: ORF3a


## 3. Clasificar mutaciones codificantes como sinónimas o no sinónimas

Para cada mutación en un gen codificante, extraemos el codón original del genoma de referencia (Wuhan-Hu-1), aplicamos la mutación y comparamos los aminoácidos resultantes. Si son iguales → **sinónima**; si son distintos → **no sinónima**.


In [17]:
# Ruta a la referencia (ajusta si usas otra)
REF_FASTA = BASE_DIR / "data" / "sars-cov-2" / "reference.fasta"

ref_record = next(SeqIO.parse(str(REF_FASTA), "fasta"))
ref_seq = str(ref_record.seq).upper()
print(f"Referencia cargada: {ref_record.id}, longitud: {len(ref_seq)} nt")

Referencia cargada: MN908947, longitud: 29903 nt


### Lógica de traducción

1. Para cada mutación, se identifica el codón afectado dentro del gen.
2. Se extrae el codón del genoma de referencia.
3. Se aplica la sustitución en la posición correspondiente dentro del codón.
4. Se traduce el codón original y el mutado.
5. Si los aminoácidos son iguales → sinónima; si no → no sinónima.

**Nota**: solo clasificamos mutaciones en genes codificantes (excluimos "intergenic"). Los genes en la hebra negativa no aparecen en este GFF (todos son +), así que no hay que preocuparse por complementos reversos.


In [18]:
CODON_TABLE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L',
    'CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M',
    'GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S',
    'CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W',
    'CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R',
    'GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

In [19]:
def classify_mutation(ref_nt, pos, alt_nt, gene, gff_df):
    """Clasifica una mutación como sinónima o no sinónima."""

    if gene == "intergenic":
        return pd.Series({"aa_ref": None, "aa_alt": None, "type": "noncoding"})

    row = gff_df[gff_df["gene"] == gene]
    if row.empty:
        return pd.Series({"aa_ref": None, "aa_alt": None, "type": "unknown"})

    start, end = row.iloc[0][["start", "end"]]

    rel_pos = pos - start
    if not 0 <= rel_pos < end - start + 1:
        return pd.Series({"aa_ref": None, "aa_alt": None, "type": "out_of_gene"})

    codon_pos = rel_pos % 3
    codon_start = start - 1 + (rel_pos // 3) * 3

    codon_ref = ref_seq[codon_start:codon_start + 3]

    if len(codon_ref) != 3:
        return pd.Series({"aa_ref": None, "aa_alt": None, "type": "incomplete_codon"})

    if codon_ref[codon_pos] != ref_nt:
        return pd.Series({"aa_ref": None, "aa_alt": None, "type": "ref_mismatch"})

    codon_alt = codon_ref[:codon_pos] + alt_nt + codon_ref[codon_pos + 1:]

    aa_ref = CODON_TABLE.get(codon_ref, "?")
    aa_alt = CODON_TABLE.get(codon_alt, "?")

    if "?" in (aa_ref, aa_alt):
        mtype = "unknown_codon"
    elif aa_ref == aa_alt:
        mtype = "synonymous"
    elif aa_alt == "*":
        mtype = "nonsense"
    else:
        mtype = "nonsynonymous"

    return pd.Series({"aa_ref": aa_ref, "aa_alt": aa_alt, "type": mtype})

In [20]:
print("Clasificando mutaciones (esto puede tomar un minuto)...")
classification = df_muts.apply(
    lambda row: classify_mutation(
        row["ref_nt"], row["position"], row["alt_nt"], row["gene"], gff_genes
    ),
    axis=1
)
df_muts = pd.concat([df_muts, classification], axis=1)

print("Distribución de tipos de mutación:")
print(df_muts["type"].value_counts(dropna=False))
print(f"\nTotal de mutaciones clasificadas: {len(df_muts)}")

Clasificando mutaciones (esto puede tomar un minuto)...
Distribución de tipos de mutación:
type
nonsynonymous    17150
synonymous        5564
noncoding          942
nonsense             7
Name: count, dtype: int64

Total de mutaciones clasificadas: 23663


### Resumen de mutaciones por gen y tipo

Ahora podemos ver, por gen, cuántas mutaciones son sinónimas y cuántas no sinónimas.

In [21]:
# Tabla cruzada: gen x tipo
crosstab = pd.crosstab(df_muts["gene"], df_muts["type"])
crosstab["total"] = crosstab.sum(axis=1)
crosstab = crosstab.sort_values("total", ascending=False)
print("Mutaciones por gen y tipo:")
print(crosstab.to_string())

# Proporción de no sinónimas por gen
if "nonsynonymous" in crosstab.columns and "synonymous" in crosstab.columns:
    crosstab["frac_nonsyn"] = (
        crosstab["nonsynonymous"] / (crosstab["nonsynonymous"] + crosstab["synonymous"])
    ).round(3)
    print("\nProporción de mutaciones no sinónimas por gen:")
    print(crosstab[["nonsynonymous", "synonymous", "frac_nonsyn"]].sort_values("frac_nonsyn", ascending=False).to_string())

Mutaciones por gen y tipo:
type        noncoding  nonsense  nonsynonymous  synonymous  total
gene                                                             
S                   0         0           9078         615   9693
ORF1a               0         0           3071        2480   5551
ORF1b               0         0           1482        1063   2545
N                   0         0           1090         313   1403
M                   0         0            801         253   1054
intergenic        942         0              0           0    942
ORF3a               0         0            405         271    676
ORF6                0         0            256         268    524
ORF9b               0         0            383          15    398
ORF7b               0         0            142         221    363
E                   0         0            235           3    238
ORF7a               0         0            161          44    205
ORF8                0         7             46   

In [22]:
out_classified = OUT_DIR / "mutaciones_clasificadas.tsv"
df_muts.to_csv(out_classified, sep="\t", index=False)
print(f"Guardado: {out_classified}")

# Resumen por gen para uso posterior
out_summary = OUT_DIR / "resumen_por_gen.tsv"
crosstab.to_csv(out_summary, sep="\t")
print(f"Guardado: {out_summary}")

Guardado: /export/space3/users/mariosb/LCG/BIOPYTHON/analisis_var_gene/data/processed/mutaciones_clasificadas.tsv
Guardado: /export/space3/users/mariosb/LCG/BIOPYTHON/analisis_var_gene/data/processed/resumen_por_gen.tsv
